# Colab T4 parity notebook — Safe from Fine-Tune to Phone (Sushovit)

Runtime: **Colab free T4** (Runtime → Change runtime type → T4 GPU). Everything here is **read-only**: nothing is pushed anywhere.

What it does: (1) pins the same library versions as the laptop (`ENV.md`); (2) logs in with the *read* token typed at runtime; (3) repeats smoke test G.1 (FP16 load, official template, Hinglish question) and G.2 (NF4, FP16 compute) on the T4; (4) scores the **same 300 MedMCQA dev questions** (embedded below) with the original model in FP16 — the laptop got **42.67 %**; (5) prints one JSON block at the end: paste it back and it gets checked against the local numbers.


In [1]:
!nvidia-smi
# Same pins as ENV.md (laptop). Colab's torch stays (T4 needs the CUDA 12.x wheel Colab ships).
!pip install -q "transformers==5.22.1" "peft==0.21.2" "bitsandbytes==0.50.2" "accelerate==1.15.0" "datasets==5.1.0" "auditkit[transformers]==1.1.3"
import torch, transformers, peft, bitsandbytes, accelerate
print('torch', torch.__version__, 'cuda', torch.version.cuda, '| transformers', transformers.__version__, '| peft', peft.__version__, '| bnb', bitsandbytes.__version__, '| accelerate', accelerate.__version__)
print('GPU:', torch.cuda.get_device_name(0), 'bf16 native:', torch.cuda.is_bf16_supported())


Tue Nov  3 05:24:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from getpass import getpass
from huggingface_hub import login
login(token=getpass('HF READ token (never stored): '))


In [3]:
# G.1 on the T4: FP16 load, official template with the System Preamble, greedy generation
import time, json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
RESULTS = {'platform': 'colab-t4'}
name = 'CohereLabs/tiny-aya-global'
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16, device_map='cuda')
msgs = [{'role': 'user', 'content': 'BP ki tablet ke saath paracetamol lena safe hai kya?'}]
text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
RESULTS['preamble_present'] = 'contextual safety mode' in text
RESULTS['prompt_tokens'] = len(tok(text).input_ids)
ids = tok(text, return_tensors='pt').to('cuda'); torch.cuda.reset_peak_memory_stats(); t0 = time.time()
out = model.generate(**ids, max_new_tokens=128, do_sample=False); dt = time.time() - t0
ans = tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
RESULTS['g1'] = {'peak_gb': round(torch.cuda.max_memory_allocated()/1e9, 2), 'tok_per_s': round((out.shape[1]-ids.input_ids.shape[1])/dt, 1), 'answer_head': ans[:200]}
print(json.dumps(RESULTS, ensure_ascii=False, indent=1))


config.json:   0%|          | 0.00/1.96k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/8.17k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 21.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

{
 "platform": "colab-t4",
 "preamble_present": true,
 "prompt_tokens": 379,
 "g1": {
  "peak_gb": 6.78,
  "tok_per_s": 16.0,
  "answer_head": "Paracetamol (Acetaminophen) और BP (Blood Pressure) की दवाओं के साथ लेने के बारे में सावधानी बरतने की आवश्यकता होती है। यहाँ कुछ महत्वपूर्ण बातें हैं जिन्हें ध्यान में रखना चाहिए:\n\n1. **संभावित इंटरैक्"
 }
}


In [4]:
# MedMCQA dev-300, one-token scoring (identical to scripts/eval/medmcqa_score.py on the laptop)
DEV = json.loads(r'''[{"id": "medmcqa-dev-0000", "text": "In a patient with competent lips together at rest, the lip\nline is opposite the tips of the upper incisors. The lip line is then described as\nA. Average\nB. High\nC. Incomplete\nD. Low\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0001", "text": "The most common odontogenic tumour which occurs in relation to an unerupted tooth in the anterior maxilla:\nA. Odontogenic adenomatoid tumour\nB. Odontoma\nC. Myxoma\nD. Cementifying fibroma\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0002", "text": "To preserve blood for transfusion later:\nA. Dilute with equal volume of 0.9% saline\nB. Add solution of sodium citrate\nC. Add solution of calcium chloride\nD. Add fibrinogen\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0003", "text": "A young male patient is on 5 mg haloperidol for many days, recently for last 4 days of duration he has inner restlessness and urges to move. Diagnosis is?\nA. Akathisia\nB. Tardive dyskinesia\nC. Rabbit syndrome\nD. Acute Dystonia\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0004", "text": "Main concern during prosthodontics surgery and better prosthetic prognosis?\nA. Continuity defects\nB. Minimal invasion\nC. Occlusion\nD. Cost\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0005", "text": "Major fimbrial structural component of P. gingivalis is\nA. Kgr\nB. KgP\nC. Fim A\nD. Fim C\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0006", "text": "In females adrenal glands are the exclusive source for secretion of?\nA. Estrogen\nB. Testosterone\nC. DHEAS\nD. Progesterone\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0007", "text": "Who was associated with second amalgam war?\nA. G V Black\nB. Charles bell\nC. Huggins\nD. Alfred stock\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0008", "text": "Hutchinson's triad includes?\nA. Interstitial keratitis, malformed molars, conductive deafness\nB. Interstitial keratitis, malformed molars, sensorineural deafness\nC. Conductive deafness, malformed incisors, interstitial keratitis\nD. Interstitial keratitis, malformed molars, conductive deafness\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0009", "text": "Which of the following is the most common symptom of aooiliac occlusive disease?\nA. Calf claudication\nB. Gluteal claudication\nC. Impotence\nD. Symptomless\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0010", "text": "A patient came with complaints of lower limb weakness. Examiner places one hand under the patient's heel and patient is asked to raise his other leg against downward resistance. What is the name of this test?\nA. Hoover test\nB. Waddell's test\nC. O'Donoghue test\nD. McBride test\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0011", "text": "A 20 years old woman has been brought to casualty with BP 70/40 mm Hg, pulse rate 120/min. and a positive urine pregnancy test. She should be managed by:\nA. Immediate laparotomy\nB. Laparoscopy\nC. Culdocentesis\nD. Resuscitation and Medical management\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0012", "text": "Best test for fetal maturity in a diabetic mother is :\nA. L:S ratio\nB. Lecithin-cephalin ratio\nC. Phosphatidyl choline\nD. Phosphatidyl glycerol\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0013", "text": "There is a swelling in a patient with respect to maxillary LI region, since last 48 hours, the swelling is hot & palpable & rebounds on pressure, the treatment of choice is\nA. Incision & drainage\nB. Antibiotic coverage\nC. Antibiotics of heat only\nD. Aspiration\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0014", "text": "The finish line used in complete metal cast crown is:\nA. Chamfer\nB. Chafer with bevel\nC. Feather edge\nD. Shoulder\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0015", "text": "Thickness of luting cement is:\nA. 20-40 μ\nB. 10-20 μ\nC. 60-80 μ\nD. 1-2 μ\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0016", "text": "The best radiograph to obtain a view of maxillary sinus is:\nA. Lateral oblique maxilla\nB. Submentovertex\nC. Waters view\nD. Lateral skull\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0017", "text": "Autogenic inhibition is afunction of\nA. Intrafusal fibre\nB. Extrafusal fibre\nC. Muscle spindle\nD. Golgi tendon organ\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0018", "text": "Which of the following drugs used for management of preterm labor for also has Neuro- protective role in fetus:-\nA. MgSO4\nB. Nifedipine\nC. Ritodrine\nD. Isoxsuprine\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0019", "text": "The drug not belonging to amide group\nA. Procaine\nB. Xylocaine\nC. Lignocaine\nD. Bupivacaine\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0020", "text": "Which is best method to assess fetal damage in a diabetes mother in 1st trimester is:\nA. Blood sugar estimation\nB. Urine ketone assay\nC. Amniocentesis to see level of sugar in amniotic fluid\nD. Glycosylated Hb\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0021", "text": "Which of the following sensation is NOT perceived by spinal nucleus of trigeminal nerve\nA. Touch\nB. Pain\nC. Temperature\nD. Proprioception\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0022", "text": "A man presents it n rashes on face and also complains of decreased mental function. He is also having few macular lesions on his skin. On CT scan, intracranial calcification was seen. His wife is normal. His 10-year old daughter is also normal but his 6-year old son is also having similar skin lesions. What would be the most likely diagnosis?\nA. Neurofibrornatosis-1\nB. Neurofibromatosis-2\nC. Xeroderma pigmentosum\nD. Autosomal dominant inheritance\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0023", "text": "Landmark for pup chamber opening is :\nA. CEJ\nB. Cervical third of crown\nC. Level of alveolar bone\nD. Enamel\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0024", "text": "V) hich of the following can change the gene expression by methylation and acetylation without affecting the content of the gene?\nA. Epigenetics\nB. Translocation\nC. Inversion\nD. Transduction\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0025", "text": "Which of the following is not used as a disinfectant?\nA. 1-2% Cetrimide\nB. 100% Alcohol\nC. 2% Lysol\nD. 5% Chloroxylene\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0026", "text": "Which periodontal fibres are consistent and are reconstructed even after the destruction of the alveolar bone?\nA. Apical\nB. Alveolar crest group\nC. Oblique\nD. Transseptal\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0027", "text": "A female come to gynaeoPD for preconceptual counseling, with history of two second trimester aboions. What is the next investigation you will advice\nA. TVS\nB. hysteroscopy\nC. Endometrial biopsy\nD. chromosomal abnormalities\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0028", "text": "If continuous hypersensitivity develops in recently restored tooth:\nA. No treatment necessary\nB. Remove the restoration and place ZOE\nC. Perform pulpectomy\nD. Perform direct pulp capping\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0029", "text": "After the delivery of an infant of diabetic mother, blood glucose of the infant was 60 mg/dt. Which other investigation docs the sister expects that the physician would ask her to do?\nA. Serum potassium\nB. CBC\nC. Serum calcium\nD. Serum chloride\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0030", "text": "Metal coping seen in upper anteriors is suggestive of:\nA. Cast post & core\nB. PFM crowns\nC. Metal coping with high stiffness\nD. Metal coping with low stiffness\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0031", "text": "Duration of action of proparacaine:\nA. 2min\nB. 5min\nC. 10min\nD. 20min\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0032", "text": "What is the most common tumor of mediastinum?\nA. Neurogenic\nB. Thymoma\nC. Lymphoma\nD. Metastatic tumor\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0033", "text": "IOL placed in the young male., After 10 year what should be done..?\nA. IOL should be removed in case of Posterior Capsule ossification\nB. Never be removed\nC. Remove when presbyopia sets in\nD. Should be changed after 10 year\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0034", "text": "A layman before introduction of fluoride in prevention of dental caries knew about it as a constituent used in\nA. Pesticide\nB. Industrial use\nC. Fertilizers\nD. Irrigant\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0035", "text": "The disturbances occurred during \"Calcification\" stage of tooth development is seen in:\nA. Peg laterals\nB. Microdontia\nC. Supernumerary tooth\nD. Interglobular dentin\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0036", "text": "A study was done where the sale of anti-asthma drug was recorded and the number of deaths because of asthma was measured over a period of 15 years (1975-1990). This is which type of study.\nA. Ecological\nB. Case reference\nC. Experimental\nD. Psephology\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0037", "text": "Which of the following is a dye used for dentinal caries detection:\nA. Calcein\nB. Zygo ZL-22.\nC. Fuschin.\nD. Acid red system.\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0038", "text": "Dorsal part of 2nd pouch give rise to:\nA. Tonsils.\nB. Thymus.\nC. Tubotympanic recess.\nD. Tympanic antrum.\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0039", "text": "Which of the following is not used as an antiflux?\nA. Graphite\nB. Boric acid\nC. Iron oxide\nD. Calcium carbonate with alcohol\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0040", "text": "Which stage of plasmodium vivax is infective to mosquito?\nA. Sporozoite\nB. Gametocyte\nC. Merozoite\nD. Zygote\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0041", "text": "Which of the following is non parametric test\nA. Pearson correlation\nB. ANOVA\nC. Mann whitney test\nD. Student t test\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0042", "text": "In fixed orthodontic treatment, how much force is needed for intrusion of four mandibular anterior teeth:\nA. 50 gm\nB. 100gm\nC. 150gm\nD. 200gm\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0043", "text": "A patient presents with ascending muscle weakness for 2 days. On examination, the limb is flaccid. What investigation should be done first?\nA. Serum potassium\nB. Serum creatinine\nC. Serum magnesium\nD. Serum calcium\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0044", "text": "Most specific for diagnosing Ankylosing spondyliti?\nA. HLA B27\nB. B/l sactoilitis\nC. Lumbar movement\nD. ESR\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0045", "text": "Which of the following will not decrease mother to child transmission of HIV?\nA. Avoid breastfeeding\nB. Vaginal delivery\nC. Zidovudine given to mother antenataly and to neonate after bih\nD. Vitamin A supplementation given to mother\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0046", "text": "Methacholine acts at which receptor?\nA. M1\nB. M2\nC. M3\nD. M4\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0047", "text": "Full mouth disinfection is done in:\nA. 24 hours\nB. 48 hours\nC. 72 hours\nD. 36 hours\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0048", "text": "Which of the following tooth structure during formation\nis most effected due to Vit -A deficiency\nA. Enamel\nB. Dentin\nC. Cementom\nD. Periodontal Ligament\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0049", "text": "Which of the following sinus grows till early adulthood:-\nA. Maxillary\nB. Ethmoidal\nC. Frontal\nD. Sphenoid\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0050", "text": "The community periodontal index of treatment needs is recorded for:\nA. Quadrants\nB. Sextants\nC. One arch\nD. Whole dentition\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0051", "text": "Comprehensive emergency obstetric care does not include:\nA. Manual removal of placenta\nB. Hysterectomy\nC. Blood transfusion\nD. Cesarean section\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0052", "text": "Oncotype Dx test is done to for the following in breast cancer:\nA. Chemotherapy in hormone receptor positive patients\nB. Hormone therapy in hormone positive\nC. Chemotherapy in hormone receptor negative patients\nD. Herceptin in Her-2-neu +ve.\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0053", "text": "Modulus of elasticity of which cement is best to support under complex amalgam restoration?\nA. GIC\nB. ZnPO4\nC. ZOE\nD. Polycarboxylate\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0054", "text": "A 7-year old boy presented with severe abdominal pain. On examination, he had xanthoma. Blood sample was taken for work-up blood sample had milky appearance of plasma. Which of the following lipoprotein is increased?\nA. LDL\nB. HDL\nC. Chylomicron\nD. Chylomicron remnants\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0055", "text": "There was an outbreak of MRSA in the hospital and it was found that a nurse of NICU had MRSA colonisation of anterior nares. What is the best treatment?\nA. Topical bacitracin\nB. Oral Vancomycin\nC. Inhaled colistin\nD. IV cefazolin\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0056", "text": "The most mucostatic impression material is\nA. Thin mix of plaster of paris\nB. Zinc oxide eugenol impression paste\nC. Free flowing wax\nD. Reversible hydrocolloids\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0057", "text": "Types of pocket respectively:\nA. Simple pocket, compound pocket, complex pocket.\nB. Simple pocket, complex pocket, compound pocket.\nC. Compound pocket, simple pocket, complex pocket.\nD. Simple pocket, compound pocket, tertiary pocket.\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0058", "text": "When adequate bone  buttressing is present, the AO system of fixation suggests\nA. Compression Osteogenesis\nB. Load bearing fixation\nC. Load sharing fixation\nD. Non-rigid fixation\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0059", "text": "Mid palatal split following Le-Fort 1 osteotomy is done by moving the maxilla in which direction?\nA. Superior\nB. Inferior\nC. Antero-posterior\nD. Transverse\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0060", "text": "Finishing and Polishing of Amalgam make the restoration:\nA. Increase in tarnish and corrosion resistance\nB. Increase the marginal strength\nC. Decrease the tarnish and corrosion resistance\nD. Increased compressive strength\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0061", "text": "Microdontia is most commonly seen affecting:\nA. Max. Lateral incisor\nB. Mand. second premolar\nC. Mand. central incisor\nD. Mand. first premolar\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0062", "text": "Half-life of TGF -B (beta) is\nA. 24 hrs\nB. 4a hrs\nC. 10-12 min\nD. 3-5 min\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0063", "text": "Fourth generation apex locater based on\nA. Resistance\nB. lmpedence\nC. Frequency\nD. Multiple frequency ratio\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0064", "text": "Investigation of choice to rule out biliary atresia in a 2-month-old child is:\nA. Hepatic scintigraphy\nB. ERCP\nC. USG\nD. CT scan.\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0065", "text": "A middle aged lady was found in a robbed room lying in a pool of blood on forensic examination there was an entry wound of size around 2 x 2 cm on the left temporal region with tattooing and blackening around the wound. There was also an exit wound with beveling at the right temporal region. On fuher examination two bullet fragments were found inside the brain parenchyma. Which of the following could be used to determine the distance from which the weapon was fired?\nA. Hair\nB. Clothes\nC. Bullet fragments\nD. Blood\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0066", "text": "Acetyl choline release can be increased from presynaptic membrane by:\nA. Blocking voltage gated Na+ channels on presynaptic membrane\nB. Blocking voltage gated Ca+ channels on presynaptic membrane\nC. Blocking voltage gated K+ channels on presynaptic membrane\nD. Blocking voltage gated CI- channels on presynaptic membrane\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0067", "text": "Transformation into osteosarcoma is seen with\nA. Pagets disease and polyostotic fibrous dysplasia\nB. Pagets disease and osteopetrosis\nC. Cherubism and polyostotic fibrous dyspLasia\nD. Cherubism and polyostotic fibrous dyspLasia\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0068", "text": "In dental composite 75 % by weight filler is approximately what % volume of filler?\nA. 85\nB. 90\nC. 60\nD. 50\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0069", "text": "A 55- year old diabetic patient presents with transient obscuration in vision for 2-3 days followed by sudden loss of vision. Which of the following would be the best test to evaluate the symptoms?\nA. Serum ACE levels\nB. Quantiferon-Gold TB test\nC. Elevated homocysteine levels\nD. Serum creatinine levels\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0070", "text": "Which of the following is not done for antenatal diagnosis of Down's syndrome:\nA. Amniotic fluid volume estimation\nB. Alpha-fetoprotein estimation\nC. Cordocentesis\nD. Chorionic villous biopsy\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0071", "text": "\"Multifactorial Causation\" theory was given by\nA. McMohan Pugh\nB. Pettenkofer of Munich\nC. Winslow\nD. Robert Koch\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0072", "text": "Premaxilla is derived from\nA. Maxillary protuberance\nB. Palatine bones\nC. Frontonasal process\nD. Median process\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0073", "text": "In order to have balance in hand instrument:\nA. Blade edge should be within 2 mm of the axis of handle\nB. Blade edge should be within 2 mm of the axis of shank\nC. Blade edge should be within 2.5 mm of the axis of handle\nD. Blade edge should be within 2.5 mm of the axis of shank\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0074", "text": "A two year old girl child is brought to the out patient with features of hand wringing stereotype movements, impaired language and communication development, breath holding spells, poor social skills and deceleration of head growth after 6 months of age. The most likely diagnosis is –\nA. Asperger's syndrome\nB. Rett's syndrome\nC. Fragile x–syndrome\nD. Colarad syndrome\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0075", "text": "Treatment of solitary langerhans histiocytoma of mandible is by\nA. Curettage\nB. Radiotherapy\nC. No treatment required\nD. Chemotherapy\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0076", "text": "For a loose complete denture relining and rebasing can be done in which of the following condition\nA. Excessive alveolar bone loss\nB. When centric relation and centric occlusion do not coincide\nC. The patient is poor and cannot afford new denture\nD. More than 2mm loss of alveolar bone height\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0077", "text": "leksell efferents go to\nA. Interneurons\nB. Spindle\nC. Golgi tendon organ\nD. Golgi bottle cells\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0078", "text": "A child is admitted on 7 days of life with severe respiratory distress and shock. He was discharged 2 days back healthy. What could be the probable diagnosis –\nA. VSD large\nB. Hypoplastic left heart syndrome\nC. Ebstein anomaly\nD. AP window defect\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0079", "text": "12 years old Shyam presented with gross hematuria with 80% dysmorphic RBC's 2 days after a attack of upper respiratory tract infection diagnosis is –\nA. Microangiopathic thrombotic anaemia\nB. IgA Nephropathy\nC. PSGN\nD. H.S. purpura\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0080", "text": "Statistical `Q' test is used for:\nA. Comparing the propoion of means of 2 groups\nB. To determine outliars\nC. To determine normality distribution\nD. Comparing the propoion of means of more than 2 groups\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0081", "text": "Presented edema, oliguria and frothy urine. He has no past history of similar complaints. On examination, his urine was positive for 3+ proteinuria, no RBCs/WBCs and no casts. His serum albumin was 2.5 gm/L and serum creatinine was 0.5 mg/dL. The most likely diagnosis is:\nA. Minimal change disease\nB. IgA nephropathy\nC. Interstitial nephritis\nD. Membranous nephropathy\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0082", "text": "A case of jaundice with 50% direct bilirubin, other LFTs normal. Diagnosis is –\nA. Rotor syndrome\nB. Gilbert syndrome\nC. Glucuronyl transferase deficiency\nD. Primary biliary cirrhosis\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0083", "text": "Erosive lichen planus resembles which of the following\nA. Monilial gingivitis\nB. Desquamative gingivitis\nC. Herpetic gingivitis\nD. Acute ulcerative gingivitis\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0084", "text": "Which of the following should be treated by using a \ntissue conditioner material?\nA. Tissue symptoms related to diabetes mellitus\nB. Generalized ridge hyperplasia\nC. Traumatized mucosa caused by ill fitting dentures\nD. Sharp residual ridge in molar area\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0085", "text": "Resilient silicone denture lining materials are superior \nto plasticized acrylic resin as:\nA. Bond strength with a denture is increased\nB. They remain resilient for a short time\nC. They do not support fungal growth\nD. Tissue adaptation is better\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0086", "text": "Concentration of adrenaline in local anaesthetic solution for achieving haemostasis & vasoconstriction at surgical site?\nA. 1:20000\nB. 1:50000\nC. 1:80000\nD. 1:100000\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0087", "text": "Which of the following is activated by intrinsic or extrinsic pathway's?\nA. Necroptosis\nB. Apoptosis\nC. Necrosis\nD. Pyroptosis\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0088", "text": "________occurs whenever there is an imbalance between myocardial oxygen supply and demand.\nA. Sudden Death\nB. Stable angina\nC. Unstable angina\nD. Myocardial infarction\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0089", "text": "Which of the following is not a branch of external carotid artery\nA. Posterior auricular artery\nB. Occipital artery\nC. Maxillary artery\nD. Ophthalmic artery\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0090", "text": "female patient came with right side hearing loss, better heard in a noisy environment,Audiogram shown with about 30-40 dB gap between AC-BC of right & left ear. Rinne's test negative, Weber's test centralised. Which of the following condition shown?\nA. Meniere's disease\nB. stapedial otosclerosis\nC. Presbycusis\nD. Vestibular schwannoma\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0091", "text": "Lamina cribrosa is a modification of :\nA. Sclera\nB. Choroid\nC. Optic nerve sheath\nD. Retina\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0092", "text": "Which of the following features is not seen in Crouzon syndrome?\nA. Midface hypoplasia\nB. Syndactyly\nC. Beak shaped nose\nD. Ocular hypertelorism\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0093", "text": "10 year old present with ankle pain. X-ray has Lytic Lesion with sclerotic rim at calcaneum. Following HPE findings were seen. What is your diagnosis:\nA. Eumycosis\nB. PVNS\nC. Ochronosis\nD. Hemophilic pseudotumar\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0094", "text": "Fourth generation meth-acrylate based resin sealer include?\nA. Endorez\nB. Resilon\nC. Hydron\nD. Real Seal E\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0095", "text": "An 8 year old child got trauma and got his central incisor avulsed 20 minutes back which got contaminated with debris. The child rushes to dental clinic with avulsed tooth, the treatment is:\nA. Clean root surface with saline and reimplant\nB. Clean and curette root surface and reimplant\nC. Clean root surface, do RCT and reimplant\nD. Scrubbing of root surface and reimplant\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0096", "text": "Which of the following is not associated with gingival Lesions?\nA. Herpes\nB. Recurrent apthae\nC. Pemphigus\nD. Pyugenic granuloma\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0097", "text": "Mucocutaneous circumoral pigmentation is found in:\nA. Peutz-Jeghers syndrome\nB. Plummer-Vinson syndrome\nC. Lead poisoning\nD. Bechet's syndrome\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0098", "text": "The cyst that moves by protruding the tongue is:\nA. Thyroglossal cyst\nB. Median rhomboid cyst\nC. Ranula\nD. Tracheal cyst\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0099", "text": "A 59-year-old female presents with gingival condition punched out crater-like depression covered with membrane which can be pulled off painlessly?\nA. Diphtheritic lesion\nB. Acute herpetic gingivostomatitis\nC. ANUG\nD. Syphilitic lesion\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0100", "text": "In rheumatoid arthritis there is inflammation of:\nA. Cartilage\nB. Synovial membrane\nC. Sclerosis of joints\nD. Articular bone\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0101", "text": "Centre of rotation during intrusion is at:\nA. Infinity\nB. Middle third of tooth\nC. CEJ\nD. Outside the tooth\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0102", "text": "In a population of 10000 people, the prevalence of a disease is 20%. The sensitivity of a screening test is 95% and specificity is 80%. The positive predictive value of the test will be -\nA. 45.70%\nB. 54.30%\nC. 15.30%\nD. 98.50%\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0103", "text": "Which of the following is unlikely to cause enamel hypoplasia?\nA. Rickets\nB. Fluoride\nC. Congenital syphilis\nD. Cleidocranial dysostosis\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0104", "text": "A 29 yrs old woman with a pregnancy of 17 week has a 10 years old boy with down syndrome. She does not want another down syndrome kid; best advice to her is\nA. No test is required now as her age is below 35 years\nB. Ultra sound at this point of time will definitely tell her that next baby will be down syndromic or not\nC. Amniotic fluid samples plus chromosomal analysis will definitely tell her that next baby will be down syndromic or not\nD. blood screening at this point of time will clear the exact picture\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0105", "text": "A young patient has been admitted with A and had massive hemorrhage. He needs to be transfused with large amounts of fluids. Which IV cannula is preferred?\nA. Grey\nB. Green\nC. Blue\nD. Pink\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0106", "text": "Highly vascular tumor of brain and spinal cord in adults:\nA. Metastasis\nB. Pilocytic astrocytoma\nC. Hemangioblastoma\nD. Cavernous malformation\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0107", "text": "During post preparation in distal canal of mandibular first molar, maximum chance of perforation is seen in:\nA. Lingual surface\nB. Distal surface\nC. Buccal surface\nD. Mesial surface\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0108", "text": "Which of the following conversions does not require Biotin as a cofactor?\nA. Gamma carboxylation of glutamate\nB. Acetyl Co-A to Malonyl Co-A\nC. Propionyl Co-A to methyl malonyl Co-A\nD. Pyruvate to oxaloacetate\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0109", "text": "A young man was on high protein diet and raw eggs. After 3 days he developed weakness. Blood investigation revealed hypoglycemia. Hypoglycemia is due to inhibition of which of the following enzymes\nA. Glucose 6 phosphatase\nB. Glycogen phosphorylase\nC. Pyruvate Carboxylase\nD. Glucokinase\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0110", "text": "A patient comes to your clinic with a complaint of multiple episodes of loose watery stool for 3 days. On probing, you discover that these episodes sta after he had ingested shellfish at a local restaurant 3 days back and other people who had food from that restaurant had similar symptoms. What is the most common cause of viral diarrhoea in adults?\nA. Calicivirus\nB. Rotavirus\nC. Adenovirus\nD. Norovirus\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0111", "text": "The most common intraoral location for a pigmented\nnevi is the\nA. Hard palate\nB. Soft palate\nC. Buccal mucosa\nD. Floor of mouth\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0112", "text": "Which Hodgkin’s disease is associated with best prognosis?\nA. Lymphocytic predominance\nB. Lymphocyte depletion\nC. Mixed cellularity\nD. Nodular sclerosis\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0113", "text": "The radiographic feature of sinusitis includes\nA. Fluid Levels\nB. Erosion of bone\nC. Clouding of antra\nD. Clouding and fluid level\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0114", "text": "The most common gene defect in idiopathic steroid resistant nephrotic syndrome –\nA. ACE\nB. NPHS 2\nC. HOX 11\nD. PAX\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0115", "text": "Increased depth in posterior palatal seal area of denture \ncauses:\nA. Tingling sensation\nB. Gagging\nC. Increased retention\nD. Unseating of denture base\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0116", "text": "Among the following solvent which of the following is safest and effective gutta percha solvent?\nA. Terpentine\nB. Choloroform\nC. Methylchioroform\nD. Halothane\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0117", "text": "Enzyme regulating the conversion of ethanol to acetaldehyde:\nA. Alcohol dehydrogenase\nB. Acetaldehyde dehydrogenase\nC. Catalase\nD. Enolase\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0118", "text": "For which of the following, you will not perform pulpectomy?\nA. Primary teeth with sinus tract\nB. Primary teeth without successor\nC. Primary teeth with furcation pathology\nD. Pulpless primary teeth next to the line of a palatal cleft\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0119", "text": "In mechanical trituration of amalgam, what is adversely affected ?\nA. Hardness of filling\nB. Tarnish resistance\nC. Working time\nD. Final gloss of the filling\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0120", "text": "Which of these is most commonly used as pre-anesthetic medication?\nA. Atropine\nB. Promethazine\nC. Scopolamine\nD. Glycopyrrolate\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0121", "text": "A person after consuming raw eggs presents with weakness, fatigue & hypoglycemia. Doctor gave him vitamin tablets. Which enzyme deficiency is causing hypoglycemia in this patient:\nA. Phosphoenol pyruvate carboxykinase\nB. Pyruvate carboxylase\nC. Glycogen phosphorylase\nD. Glucose 6 phosphatase\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0122", "text": "Topical administration of I% Pilocarpine failed to produce pupillary constriction in a patient who had a large, dilated pupil What should be the most probable reason?\nA. Adie's tonic pupil\nB. Diabetic III nerve palsy\nC. Pharmacological blockade\nD. Uncal herniation\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0123", "text": "Most abundant cells in Cell rich zone\nA. Fibroblasts\nB. Odontoblasts\nC. Ameloblasts\nD. Undifferentiated mesencymal cells\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0124", "text": "Numbness on shaving after parotidectomy due to which nerve injury?\nA. Facial nerve\nB. Auriculotemporal nerve\nC. Greater auricular nerve\nD. Occipital nerve\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0125", "text": "A 5–years old male child presents with episodic anaemia and jaundice since birth. He is least likely to have which of the following\nA. Hereditary spherocytosis\nB. Siclde cell anemia\nC. PNH\nD. G–6–PD deficiency\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0126", "text": "GNAS-1 gene mutation is associated with:\nA. Fibrous dysplasia\nB. Osteogenesis imperfecta\nC. Amelogenesis imperfecta\nD. Cleidocranial dysplasia\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0127", "text": "Which of the following is not characteristic feature of Binder syndrome?\nA. Acute naso-labial angle\nB. Flat nasal bridge\nC. Class III malocclusion\nD. Class II malocclusion\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0128", "text": "Function of Merkel's cells is:\nA. Tactile sensation\nB. Melanophage\nC. Chemoreceptor\nD. Proprioception\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0129", "text": "Injury to the right mandibular nerve as it passes through foramen ovale produces the following effects:\nA. Loss of taste from the anterior 2/3 of tongue\nB. Deviation of the tongue to the left\nC. Deviation of chin to left when mouth is opened\nD. Loss of general sensation to anterior 2/3 of tongue\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0130", "text": "Which of the following nucleus has cardio inhibitory function?\nA. Nucleus Ambiguus\nB. Nucleus Tractus Solitarius\nC. Rostral Ventrolateral Medulla\nD. Dorsal motor nucleus of vagus\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0131", "text": "A 48 years old female presents with seizure, recurrent gross hematuria and left flank abdominal pain. Abdominal CT reveals left perinephric hematoma with 3 cm angiomyolipoma along with multiple right renal angiomyolipoma measuring 1.5 to 6.5 cm. What would be the most probable diagnosis?\nA. VHL syndrome\nB. Autosomal dominant polycystic kidney disease\nC. Tuberous sclerosis\nD. Hereditary angiolipoma\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0132", "text": "MB2 (In maxillary first molar)can be seen through Dental operating microscope in what percentage of cases\nA. 90-95%\nB. 70-85%\nC. 62-70%\nD. 30-40%\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0133", "text": "Renal agenesis is associated with:\nA. Hydramnios\nB. Anencephaly\nC. Tracheo-oesophageal fistula\nD. Oligohydramnios\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0134", "text": "Composition of KRI paste is:\nA. Calcium hydroxide + Iodoform.\nB. Iodoform+ calcium hydroxide + menthol.\nC. Iodoform + camphor + parachlorophenol + Menthol.\nD. Parachlorophenol + Camphor + menthol.\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0135", "text": "Which of the following anti-tubercular drug can cause ophthalmological toxicity?\nA. Isoniazid\nB. Rifampicin\nC. Ethambutol\nD. Pyrazinamide\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0136", "text": "In fracture of atrophic mandible with bone loss, what is the best treatment modality?\nA. Bone grafting and load bearing\nB. Bone grafting and load sharing\nC. Semi-rigid fixation\nD. IMF with open reduction\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0137", "text": "Percentage of Phosphoric acid where Dicalcium phosphate\nmonohydrate is formed that cannot be rinsed off.\nA. 25%\nB. 37%\nC. 50%\nD. 85%\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0138", "text": "Sickle cell anemia is a -\nA. Gene deletion\nB. Gene modification\nC. Point mutation\nD. Frame shift mutation\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0139", "text": "Placenta grade 3, 35+3 weeks pregnancy, and absent end diastolic flow Doppler; next ,management is:-\nA. Dexamethasone and terminate after 48 hours\nB. Terminate after 37 weeks\nC. Talk with pediatrician and termination\nD. Monitor\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0140", "text": "Gait apraxia is seen in thromboembolic episode involving:\nA. ACA\nB. MCA\nC. PCA\nD. Posterior choroidal aery\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0141", "text": "Which of the following drug is used to counter act the gastric irritation produced by administration of NSAID:\nA. Roxatidine\nB. Pirenzipine\nC. Betaxolol\nD. Misoprostol\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0142", "text": "Beveling of the pulpoaxial line angle of a class II cavity\nis done to:\nA. Increase the strength of the restoration\nB. Improved marginal adaptation\nC. To prevent the fracture of enamel\nD. To prevent the fracture of amalgam\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0143", "text": "Retraction of mandible is achieved by:\nA. Lateral pterygoid\nB. Temporalis\nC. Medial pterygoid\nD. Masseter\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0144", "text": "Ganglionic transmission is mediated by:\nA. Presynaptic alpha-receptors\nB. Postsynaptic beta-receptors\nC. Postsynaptic dopaminergic receptors\nD. Postsynaptic nicotinic receptors\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0145", "text": "Sudden death in a patient with an acute myocardial\ninfarction is most commonly due to\nA. Aortic dissection\nB. Atrial fibrillation\nC. Cardiac tamponade\nD. Ventricular fibrillation\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0146", "text": "What is the usual weight of rabbit used in ophthalmological experiments?\nA. 0.5-1 kg\nB. 1.5-2.5 kg\nC. 5-7 kg\nD. 10-12 kg\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0147", "text": "On laboratory investigations in a patient. pH = 7.3, pCO2 = 35 mm Hg. What is the likely acid base imbalance?\nA. Respiratory acidosis\nB. Metabolic acidosis\nC. Metabolic alkalosis\nD. Respiratory alkalosis\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0148", "text": "A Middle aged woman presents with fissures in mouth, tingling sensation and peripheral neuropathy. Investigations showed reduced glutathione reductase activity. which vitamin deficiency is the likely cause of this?\nA. Vitamin B1\nB. Vitamin 82\nC. Vitamin 86\nD. Vitamin B l2\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0149", "text": "Which of the following drug is commonly used in treatment for cancer associated thromboembotismt\nA. Low molecular weight heparin\nB. anti-thrombin III inhibitors\nC. Direct Xainhibitors\nD. Warfarin\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0150", "text": "In submandibular surgery incision is given 2 cm below the border of mandible to preserve which nerve?\nA. Marginal mandibular nerve\nB. Mental Nerve\nC. Long buccal nerve branch of facial\nD. Inferior alveolar nerve\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0151", "text": "Pneumococcal vaccine PCV 23 polysaccharide has shown the best results in the following people\nA. Sickle cell anemia\nB. Cystic fibrosis\nC. Child less than 2 year age\nD. Recurrent otitis media and sinusitis\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0152", "text": "Which is not a dimorphic fungi?\nA. P. marneffi\nB. Histoplasma capsulatum\nC. Blastomyces dermatitidis\nD. Pneumocystis jirovecii\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0153", "text": "In a patient with obstructive jaundice, what is the possible explanation for a bilirubin level of 40 mg/dL?\nA. Malignant obstruction\nB. Complete obstruction of common bile duct\nC. Renal failure\nD. Liver failure\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0154", "text": "Most important prognostic factor in congenital diaphragmatic hernia –\nA. Pulmonary hypertension\nB. Size of hernia\nC. Timing of surgery\nD. Gestational age\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0155", "text": "Cartilage of larynx are developed from:\nA. 3rd arch cartilage.\nB. 4th arch cartilage.\nC. 4&5 arch cartilage.\nD. 4&6 arch cartilage.\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0156", "text": "A 35 years old female patient present with hearing loss with improvement in hearing in noisy environment. On examination Rinne's is negative and weber's is centralized. Following is the audiometry repo. What is the most likely diagnosis:\nA. Meniere's disease\nB. Perilymph fistula\nC. Stapedial Otosclerosis\nD. Vestibular Schwannoma\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0157", "text": "Which of the following is innervated by the vagus nerve?\nA. The levator veli palatine (levator palatini)\nB. The posterior belly of digastric\nC. Mylohyoid\nD. The tensor veli palatine (tensor palatini)\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0158", "text": ". The most common fungal infection in the neonates transmitted by caregiver's hand is:\nA. Candida albicans\nB. Candida glabrata\nC. Candida tropicalis\nD. Candida parapsilosis\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0159", "text": "A patient of diabetes and hypeension comes to your clinic. As a doctor, you explain to him the risks of various complications. Which of these is the best tool to demonstrate the complications?\nA. Pie cha\nB. Histogram\nC. Scatter plot\nD. Venn diagram\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0160", "text": "A patient is giving; history of avulsed tooth 20 minutes back, comes to dentist what should be done?\nA. Scrub the tooth and reimplant\nB. Rinse with saline and reimplant\nC. Sterilize tooth and reimplant\nD. Scrub the tooth do RCT and reimplant\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0161", "text": "Serum glutamic oxaloacetic transaminase is increased in:\nA. Renal failure\nB. Prostatic carcinoma\nC. Myocardial infarction\nD. Pulmonary edema\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0162", "text": "The resolution of IOPA is about:\nA. 10 Lp/mm\nB. 16 Lp/mm\nC. 28 Lp/mm\nD. 30 Lp/mm\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0163", "text": "Amyloid protein in Hemodialysis associated with amyloidosis is-\nA. AA\nB. A b\nC. b- 2 microglobulin\nD. Transthyretin\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0164", "text": "Molecule with permanent dipole\nA. Oxygen\nB. Helium\nC. Liquid nitrogen\nD. Water\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0165", "text": "Characteristic histopathological feature of basal cell car cinoma is\nA. Nuclear palisading\nB. Keratin pearls\nC. Psammoma bodies\nD. Foam cells\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0166", "text": "Rabies vaccine not to be given in case of?\nA. Rat bite\nB. Cat bite\nC. Dog bite\nD. Monkey bite\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0167", "text": "Which of the following is the complication of massive blood transfusion?\nA. Metabolic acidosis\nB. Metabolic alkalosis\nC. Respiratory alkalosis\nD. Respiratory acidosis\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0168", "text": "Eugenol may be replaced in the zinc oxide eugenol cement by:\nA. Acetic acid\nB. Alginic acid\nC. Phosphoric acid\nD. Ortho-ethoxy acid\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0169", "text": "The anteroposterior curve is\nA. Curve of Spee\nB. Curve of Wilson\nC. Curve of Monson\nD. Bonwill's curve\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0170", "text": "Mechanism of adhesion of GIC restoration with tooth surface is by means of?\nA. Carboxyl group\nB. Chelates with Metal Ions\nC. C=C double bond\nD. Polymer Chains\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0171", "text": "In human the Hb is :\nA. HbH\nB. HbA\nC. HbM\nD. HbS\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0172", "text": "The decreased phosphate levels seen in hyper parathyroidism is due to:\nA. Decreased intestinal phosphate absorption\nB. Increased calcium excretion\nC. Decreased renal phosphate absorption\nD. Increased loss of phosphate in urine\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0173", "text": "Which of the following enzyme is required to induce the formation of a plasma clot pathogenic staphylococci\nA. Lysozyme\nB. Fibrinolysin\nC. Coagulase\nD. S protein\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0174", "text": "Reversal lines are characteristic on the radiograph of:\nA. Osteoma\nB. Osteosarcoma\nC. Epidermoid cyst\nD. Paget's disease of bone\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0175", "text": "Boiled lobster syndrome is seen in poisoning of:\nA. Boric acid\nB. HNO,\nC. H,SO4\nD. Phenol\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0176", "text": "A newborn presents with congestive heart failure, on examination has bulging anterior fontanelle with a bruit on auscultation. Transfontanellar USG shows a hypoechoic midline mass with dilated lateral ventricles. Most likely diagnosis is –\nA. Medulloblastoma\nB. Encephalocele\nC. Vein of Galen malformation\nD. Arachnoid cyst\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0177", "text": "plate endings are features of\nA. Nuclear chain fibres\nB. Nuclear bag fibres\nC. Golgi tendon\nD. Extra fusal\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0178", "text": "Contractile element in myofibril is:\nA. H band\nB. Sarcoplasm\nC. Sarcomere\nD. A line\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0179", "text": "Teeth selected for scoring of PHP-M (Personal Hygiene Performance Modified) are:\nA. Six\nB. Twelve\nC. Eighteen\nD. Twenty four\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0180", "text": "Etanercept is a disease-modifying drug used in management of rheumatoid ahritis. What is its mechanism of action?\nA. Inhibition of TNF alpha\nB. COX-2 inhibition\nC. IL-6 inhibition\nD. Stabilization of mast cells\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0181", "text": "Major determinant of loading dose of a drug is:-\nA. Half life\nB. Clearance\nC. Volume of distribution\nD. Bioavailability\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0182", "text": "Which tooth is known as enigma to endodontist\nA. Mandibular canine\nB. Maxillary 1st premolar\nC. Mandibular 2nd premolar\nD. Mandibular 1st premolar\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0183", "text": "Reactive oxygen metabolite in lysosomes is released by\nA. Peroxidase\nB. NADPH oxidase\nC. Superoxide dismutase\nD. Catalase\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0184", "text": "The highest (longest) and sharpest cusp on the deciduous mandibular first molar is:\nA. Mesiolingual\nB. Distolingual\nC. Mesiobuccal\nD. Distobuccal\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0185", "text": "Insufficient closure of the flasks during packing of denture base resin can result in:\nA. Poor colour stability\nB. Less polymerization shrinkage\nC. Increase in vertical dimension\nD. Decrease in vertical dimension\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0186", "text": "A girl comes with symptoms of involuntary movements. Sydenham's chorea and acute rheumatic fever is suspected. Other major criteria of rheumatic fever (ahritis, skin rashes, subcutaneous nodules and carditis) were absent. No evidence of sore throat. Best investigation to prove rheumatic etiology is:\nA. Antistreptolysin S\nB. Antistreptolysin O\nC. Throat culture\nD. PCR for M protein\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0187", "text": "Chloroquine related eye disease risk is\nA. HIV protease\nB. HIV reverse transcriptase\nC. HIV integrase\nD. HIV entry into the cell\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0188", "text": "Which of the following hea sound may be a normal finding during pregnancy?\nA. Fixed splitting of S2\nB. S3\nC. S4\nD. Pericardial knock\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0189", "text": "In a group of 100 people, the average GFR is 85 ml/ min with a standard detion of 25. What is the range for 90% confidence interval?\nA. 81-89\nB. 80-90\nC. 75-95\nD. 70-100\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0190", "text": "The non-invasive method to measure the blood flow is:\nA. Electric pulp test\nB. Percussion\nC. Radiograph\nD. Laser Doppler flowmetry\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0191", "text": "Acromegaly is associated with\nA. Class. I malocclusion\nB. Class. I cross bite\nC. Class. II malocclusion\nD. Class. III malocclusion\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0192", "text": "Shape of trapezium muscle is\nA. Triangular\nB. Quadrilateral\nC. Trapezium\nD. Quadrangular\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0193", "text": "NESTROFT test is a screening test for –\nA. β–thalassemia\nB. Hereditary spherocytosis\nC. Autoimmune hemolytic anemia\nD. Megaloblastic anemia\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0194", "text": "After I.V. administration of which of the following anesthetic agent, there is rapid recovery and less headedness?\nA. Propofol\nB. Diazepam\nC. Droperidol\nD. Midazolam\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0195", "text": "A 70 years old male patient presents with amblyopia, exeional chest pain, episodic tachycardia and extra systoles on FCC. What is the probable cause?\nA. Cocaine poisoning\nB. Chronic nicotine poisoning\nC. Arsenophagia\nD. Cannabis ingestion\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0196", "text": "Inferior alveolar nerve runs:\nA. Medial to lateral pterygoid\nB. From Infratemporal fossa\nC. Lateral to medial pterygoid\nD. Lateral to lateral pterygoid\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0197", "text": "Major mechanism of action of nitrates in acute attack of angina is:\nA. Coronary vasodilation\nB. Decrease in preload\nC. Decrease in afterload\nD. Decrease in hea rate\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0198", "text": "Not included in the human poverty index is:\nA. % of population not surviving up to 40 yrs age\nB. Occupation\nC. Underweight for age\nD. % population not using safe water supply\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0199", "text": "Which of the following arteries does not supply the circle of Willis?\nA. Anterior cerebral\nB. Middle cerebral\nC. Posterior-inferior cerebral\nD. Posterior communicating\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0200", "text": "Nitrous oxide alone is not used as a G.A agent because of\nA. Difficulty in maintaining an adequate oxygen concentration\nB. Expense of the agent and its exposure hazards\nC. Adverse effect on liver\nD. Poor analgesic property\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0201", "text": "The usual radiographic appearance of an osteosarcoma is:\nA. Discrete radiolucency with regular borders\nB. Multicystic radiolucency with a soap bubble appearance with an irregular peripheral border\nC. Sunburst pattern with radiopaque strands extending from the cortical plates\nD. Cotton wool appearance with an irregular peripheral border\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0202", "text": "Which of the following marker persists in chronic hepatitis and recurrent hepatitis?\nA. IgG Anti HbcAg\nB. HBsAg\nC. IgG Anti HBsAG\nD. Anti Hbs\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0203", "text": "Difference between mandibular and maxillary premolars is that in the mandibular premolars:\nA. The buccal and lingual cusps are more prominent\nB. The lingual cusp is sharper and longer\nC. The lingual surface rounded and less developed\nD. The buccal surface is rounded and Less developed\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0204", "text": "A middle aged female presents with slowly progressive weakness of lower limbs, spasticity and recent onset hesitancy of Micturition. On neurological examination there is evidence of dorsal myelopathy. MRI scan of spine reveals middorsal intradural contrast enhancing mass lesion. Diagnosis is:\nA. Intradural lipoma\nB. Dermoid cyst\nC. Meningioma\nD. Epidermoid cyst\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0205", "text": "Definitive treatment for the condition shown in radiograph below is?\nA. Apexogenesis\nB. Revascularisation\nC. Calcium Hydroxide apexification\nD. MTA barrier formation\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0206", "text": "A 25-year-old lady presents with sudden severe bilateral loss of vision, more so on the right side, with no perception of light. Rest of the examination including pupillary reflexes, fundus and optokinetic nystagmus are normal. She was able to touch the tips of her fingers with her right eye closed but not with her left eye closed. The most like1 tgnosis is:\nA. Optic neuritis\nB. Anterior ischemic optic neuropathy\nC. CMV retinitis\nD. Functional vision loss\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0207", "text": "Foltowing are the features of the color of normal amniotic fluid during delivery?\nA. Milky to yellowish green with mucus flakes\nB. Amber colored\nC. Clear colorless to Pale Yellow\nD. Golden color\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0208", "text": "Advantage of the submarginal falp for periradicular surgery is?\nA. Crestal bone less\nB. Bleeding lees\nC. Gingival shrinking less\nD. Flap closure easy\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0209", "text": "Langerhan’s histiocytosis was previously known as\nA. Eosinophilic granuloma\nB. Pyogenic granuloma\nC. Letterer siwe disease\nD. Histiocytosis x\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0210", "text": "Which of the following parameters is used to determine the sensitivity of vaccine due to heat?\nA. VVM\nB. VMV\nC. VCM\nD. VMM\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0211", "text": "A continuum to sedation, anxiolysis is referred to:\nA. Conscious sedation\nB. Deep sedation\nC. Minimal sedation\nD. General anaesthesia\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0212", "text": "A 10-year old boy was presented with a mass in abdomen. On imaging, the para-aoic lymph nodes were enlarged. On biopsy, starry sky appearance was seen. What is the underlying abnormality?\nA. p53 gene mutation\nB. Rb gene mutation\nC. Translocation involving BCR-ABL gene\nD. Translocation involving Myc gene\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0213", "text": "Most probable chance of developing caries in the following curve is at:\nA. Point A\nB. Point B\nC. Point C\nD. Point D\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0214", "text": "Hypereosinophilic syndrome is a disease characterized by elevated eosinophil count in the blood for at least 6 months, without any recognizable cause, with involve-ment of either hea, nervous system or bone marrow. What should be the eosinophil count for diagnosis?\nA. >500/mm3\nB. >1000/ mm3\nC. >1500/ mm3\nD. >2000/ mm3\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0215", "text": "Which of these is the best study to evaluate effect and outcome?\nA. Clinical trial\nB. Coho\nC. Case control study\nD. Cross sectional study\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0216", "text": "Which of the following is used for selecting patients with respect to potential factors that will affect the results?\nA. Systematic random sampling\nB. Simple random sampling\nC. Stratified random sampling\nD. Cluster sampling\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0217", "text": "Arrange the sequence of auditory pathway from peripheral to central: Inferior colliculus Cochlear nucleus Auditory coex Medial geniculate body\nA. B>A>D>C\nB. B>D>C>A\nC. A>D>B>C\nD. C>A>B>D\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0218", "text": "Which among the following is not a risk factor for contrast-induced nephropathy?\nA. Diabetic nephropathy\nB. High osmolar agent\nC. Obesity\nD. Dehydration\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0219", "text": "Light transmitting wedges transmit:\nA. 70-80% of incident light\nB. 80-85% of incident light\nC. 85-90% of incident light\nD. 90-95% of incident light\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0220", "text": "Highest concentration of oxygen is delivered through?\nA. Nasal cannula\nB. Venturi mask\nC. Bag and mask\nD. Mask with reservoir\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0221", "text": "On applying pressure on angle of jaw while maintaining patent airway which nerve is likely to be damaged:\nA. 6th\nB. 7th\nC. 4th\nD. 9th\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0222", "text": "Square root of pq/n indicates:\nA. Standard error of mean\nB. Standard error of difference of means\nC. Standard error of proportions\nD. Standard error of difference in proportions\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0223", "text": "Which of the following has least density?\nA. VLDL\nB. LDL\nC. HDL\nD. Chylomicrons\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0224", "text": "Preferred drug for the treatment of uncomplicated grade 2 hypeension in a 48 year old man is\nA. Chlohalidone\nB. Triamterene\nC. Spironolactone\nD. Furosemide\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0225", "text": "Most common route for metastasis of oral cancer is by:\nA. Direct extension\nB. Lymphatics\nC. Blood vessels\nD. Aspiration of tumor cells\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0226", "text": "For root fracture it is imperative to take X-ray at?\nA. 45°,\t90°\nB. 90°\nC. 90°,\t110°\nD. 45°,\t90°, 110°\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0227", "text": "An intubated patient with eye opening to pain with abnormal flexion. What is the GCS score?\nA. E2VNTM3\nB. E2VTM3\nC. E2V1M3\nD. E2VTM4\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0228", "text": "One day after complete mouth extraction, blue black spots are seen on the neck of the patients. These spots indicate:\nA. Thrombocytopenic purpura\nB. Postoperative ecchymosis\nC. Impaired blood circulation\nD. Cellulitis\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0229", "text": "GNAS-1 gene mutation is associated with\nA. Fibrous dysplasia\nB. Osteogenesis imerfecta\nC. Amelogenesis imperfect\nD. Cleidocranial dysplasia\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0230", "text": "Geniculate neuralgia is caused in the nerve\nA. VII\nB. IX\nC. X\nD. II\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0231", "text": "The radiopacity that frequently obliterate the root apices of maxillary molars is\nA. Maxillary sinus\nB. Palatine bone\nC. Zygoma and Zygomatic process of Maxilla\nD. Orbital process of Zygomatic bone\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0232", "text": "A glass factory worker presented with complaints of numbness in hands and feet, generalized weakness and constipation. Radiograph showed linear lines on metaphyses of knee and wrist joints. How will you diagnose this patient?\nA. Serum mercury levels\nB. Vitamin D levels\nC. RBCcholinesterase levels\nD. Amino levulinic acid levels in urine\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0233", "text": "Which of the following complications of stroke need not to be treated?\nA. Fever\nB. Spasticity\nC. Dysphagia\nD. Numbness\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0234", "text": "Apical foramen is found at root apex in\nA. 17-46% of cases\nB. 5-10 % of cases\nC. 10-30% of cases\nD. 100% of cases\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0235", "text": "Tachycardia in a patient in shock is response to:\nA. Hypotension\nB. Vasoconstriction\nC. CNS anoxia\nD. Anxiety\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0236", "text": "Reticulocytosis is seen in:\nA. Hemolytic anemia\nB. Megaloblastic anemia\nC. Aplastic anemia\nD. Iron deficiency anemia\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0237", "text": "A pregnant lady with persistent late, variable deceleration with cervical dilatation of 6 cm shifted to OT for surgery. Which of the following is not done in Mmn:\nA. Supine position\nB. O2 inhalation\nC. I.V. fluid\nD. Subcutaneous terbutaline\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0238", "text": "Epsilon in high copper amalgam  is?\nA. Cu6Sn5\nB. Ag3Sn\nC. Cu3Sn\nD. Ag2Hg3\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0239", "text": "Inferior thyroid artery ligation during thyroidectomy is done\nA. For recurrent Laryngeal nerve\nB. Superior laryngeal nerve\nC. Vagus nerve\nD. Glossopharyngeal nerve\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0240", "text": "Which one of the following is not used as an abrasive in the dentifrice\nA. Calcium carbonate\nB. Silica\nC. Sodium Chloride\nD. Amylase\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0241", "text": "A 24 years old college student while playing hockey injured his right knee. This patient presents after 3 months with instability of knee joint in it full extension without instability at 90 degree of flexion. The structure most commonly damaged is:\nA. Posterolatilal pa of anterior cruciate ligament\nB. Anteromedial pa of anterior cruciate ligament\nC. Posterior cruciate ligament\nD. Anterior hom of medial meniscus\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0242", "text": "Anaesthestic agent causing adrenal suppression\nA. Midazolom\nB. Thiopentone\nC. Etomidate\nD. Propofol\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0243", "text": "Percentage of carbon in carbon steel hand instruments is\nA. 1-1.2%\nB. 10-12%\nC. 0.6 to 0.8%\nD. 6 to 8%\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0244", "text": "Infection from maxillary first molar drains into\nA. Submandibular space\nB. Infratemporal space\nC. Buccal space\nD. Infraorbital space\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0245", "text": "A patient complains of numbness of lower lip. There is no history of tooth extraction . It could be:\nA. Infection\nB. Metastatic neoplasia.\nC. Dental menipulation.\nD. CNS disease.\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0246", "text": "In an adult, the spinal cord ends at the level of:\nA. Lower border of L1\nB. L2\nC. L3\nD. L4\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0247", "text": "Which of the following sites is least commonly preferred for insulin injection?\nA. Anterior thigh\nB. Lateral thigh\nC. Dorsum of arm\nD. Around umbilicus\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0248", "text": "Drug causing toxic optic neuropathy is:March 2009\nA. Pyrazinamide\nB. Chloroquine\nC. Ethambutol\nD. Tetracycile\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0249", "text": "How much hone can be stretched at a time before it fractures:\nA. 1%\nB. 2%\nC. 6%\nD. 8%\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0250", "text": "The hormone having the maximum effect on granulation wound healing is:\nA. Thyroxine\nB. Cortisone\nC. Parathormone\nD. Epinephrine\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0251", "text": "Which one of the following is not an Indication for antibiotic therapy In patients with acute abscess?\nA. Fever\nB. Deep inaccessible pocket\nC. Horizontal bone loss\nD. Regional lymphadenopathy\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0252", "text": "Growth of the maxilla in the vertical direction is due to:\nA. Growth of the alveolus\nB. Growth at sutures\nC. Growth of the cranial base\nD. Growth of the synchondrosis\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0253", "text": "Which is the best incision preferred for diaphragmatic surgery?(AIIMS May 2015, May 2014)\nA. Circumferential\nB. Radial\nC. Veical\nD. Transverse\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0254", "text": "Which of the following causes vasodilation?\nA. Thromboxane A2\nB. Prostaglandin E2\nC. Histamine\nD. Serotonin\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0255", "text": "Mother of a 10 month old child came with a complaint of non-eruption of teeth, what will be the treatment protocol?\nA. Wait and watch\nB. You will do OPG which shows developing tooth buds\nC. PA skull view which shows developing tooth buds\nD. Paediatric consultation\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0256", "text": "Best method to neasure HbA1c?\nA. lsoelectric focusing\nB. Affinity chromatography\nC. Ion exchange chromatography\nD. Electrophoresis\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0257", "text": "Nobel prize for medicine/physiology in 2018 for the discovery of:\nA. Apoptotic pathway\nB. Crispr-Cas9\nC. Negative immune regulation\nD. Molecular mechanisms controlling circadian rhythm\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0258", "text": "Ratio of mesiodistal width to height of crown of maxillary canine\nA. 8:10\nB. 10:08\nC. 6:08\nD. 8:06\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0259", "text": "At physiological pH, which of these amino acids has a positive charge?\nA. Valine\nB. Aspaic acid\nC. Arginine\nD. Isoleucine\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0260", "text": "A patient notices a well demarcated area of depapillation on his tongue which has been there for as long as he can remember. The most probable diagnosis:\nA. Median rhomboid glossitis\nB. Geographic tongue\nC. Black hairy tongue\nD. Moeller's glossitis\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0261", "text": "A child presented at 18 months of age who has never been vaccinated before. Which vaccines will you administer?\nA. DPT, OPV and MMR\nB. Pentavalent vaccine alone\nC. BCG and OPV\nD. MMR, OPV, Rotavirus\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0262", "text": "In uncontrolled diabetes mellitus, elevated triglyceride and VLDL levels are seen due to:\nA. Increased activity of lipoprotein lipase and decreased activity of hormone sensitive lipase\nB. Increased activity of hormone sensitive lipase and decreased activity of lipoprotein lipase\nC. Increase in peripheral LDL receptors\nD. Increased activity of hepatic lipase\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0263", "text": "Caries tetralogy by Newburn includes the fourth factor, which is:\nA. Temperature.\nB. Time.\nC. Frequency.\nD. Substrate.\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0264", "text": "A patient with acute history of blistering and denudation involving >30% BSA along with erosions of the lips with hemorrhagic crusting and other mucosa for few days. What is the most common triggering factor?\nA. Bacterial infection\nB. Viral infection\nC. Drug induced\nD. Idiopathic\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0265", "text": "Which of the following best describes the working time of impression material?\nA. After the start of appearing elastic properties of impression material\nB. Just after the start of appearing elastic properties of impression material\nC. The time from start of mixing till just before the start of appearing elastic properties of impression material\nD. Loss of lusture of impression material\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0266", "text": "A 76 - year - old female presented with non - healing ulcer on labia majora for 6 months measuring 2 x 3 cm with no palpable lymphadenopathy. Biopsy shows Squamous cell carcinoma. Management of this patient includes?\nA. Radical vulvectomy with sentinal LN biopsy\nB. Wide excision\nC. Simple vulvectomy\nD. Chemoradiation with resection\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0267", "text": "If a patient has gingival recession, how far should the gutta percha be trimmed back before sealing the coronat area?\nA. 1 mm below the level of the recession\nB. 1 mm below the ACJ\nC. 1 mm below the canal opening\nD. 1 mm below the roof of pulpal chamber\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0268", "text": "Niacin & riboflavin help in:\nA. Redox reactions\nB. Transamination reaction\nC. Methyl group transfer\nD. Amine group transfer\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0269", "text": "A 56yr old underwent transsphenoidal hypophysectomy for pituitary tumour. Now he has low ACTH, TSH, FSH, LH. Which of the hormone will not be given to the patient?\nA. Glucocoicoids\nB. Mineralocoicoids\nC. Levothyroxine\nD. Estradiol\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0270", "text": "Treatment of diastema because of a thick labial frenum is done:\nA. After frenectomy\nB. Before eruption of canines\nC. After eruption of canines\nD. Before frenectomy\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0271", "text": "Endo-cochlear potential is:\nA. +45 mV\nB. --45mV\nC. -60mV\nD. +85 mV\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0272", "text": "Data about recent trends of immunization in the community can be found by:\nA. Sample registration system\nB. District level health survey\nC. Rural survey\nD. Census data\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0273", "text": "The 3 embryonic germ layers: ectoderm, mesoderm and endoderm are derived from:\nA. Epiblast only\nB. Ectoderm from epiblast; mesoderm and endoderm from hypoblast\nC. Ectoderm and mesoderm from epiblast; endoderm from hypoblast\nD. Hypoblast only\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0274", "text": "Which acid does not show coagulation necrosis on contact?\nA. HC1\nB. H,SO4\nC. HF\nD. HNO3\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0275", "text": "intermediate of kreb cycle used in heme synthesis;\nA. succinyl-CoA\nB. Alpha ketoglutarate\nC. Citrate\nD. Aspaate\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0276", "text": "P3L3 came to opd with postcoital bleeding and pap positive p/v cervix hyperophied bleed on touch diagnosis -\nA. CA cervix\nB. Fibroid\nC. Cervicitis\nD. Cervical polyp\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0277", "text": "Which of the following is NOT the criteria to discharge a patient of dengue haemorrhagic fever?\nA. Afebrile for 24 hours without antipyretics\nB. Platelet count greater than 50,000 cells\nC. Return of appetite\nD. Atleast 24 hours since recovery from shock\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0278", "text": "Which of these is not a theory of TMJ ankylosis?\nA. Calcification in the soft tissues around the joint space\nB. Synovial fluid attracting calcium ions\nC. Lateral displacement of fracture condyle with fusion  with zygomatic bone\nD. Condylar burst\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0279", "text": "Which of the following is the most common site for the\noccurrence of a basal cell carcinoma?\nA. Buccal mucosa\nB. Hard palate\nC. Skin of the lower lip\nD. Dorsum of the tongue\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0280", "text": "IV ultrashort acting barbiturate:\nA. Meprobamate\nB. Pentobarbitone\nC. Thiopentone\nD. Phenobarbitone\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0281", "text": "Which of the following antibiotics is most frequently associated with pseudomembranous colitis?\nA. Ampicillin\nB. Penicillin\nC. Clindamycin\nD. Cephalosporin\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0282", "text": "Location of MB2 in maxillary first molar\nA. Distopalatal to MB1\nB. Mesiobuccal to MB1\nC. Distobuccal to MB1\nD. Mesiopalatal to MB1\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0283", "text": "During maxillary osteotomy, which of the following can be given to prevent bleeding:\nA. Acetoacetic acid\nB. Lactic acid\nC. Heparin\nD. Tranexamic acid\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0284", "text": "At 28 weeks gestation, amniocentesis reveals a AOD 450 of 0.20 which is at the top of third zone of the liley curve. The most appropriate management of such a case is:\nA. Immediate delivery\nB. Intrauterine transfusion\nC. Repeat Amniocentesis after 1 week\nD. Plasmapheresis\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0285", "text": "Which of the following is a soft tissue cyst which do not produce any radiographic changes\nA. Nasolabial cyst\nB. Nasopalatine cyst\nC. Mid alveolar cyst\nD. Palatine cyst\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0286", "text": "The cyst which is found within the bone at the junction of the globular process, the Lateral nasal process & maxillary process is:\nA. Naso-alveolar cyst\nB. Globulomaxillary cyst\nC. Naso palatine cyst\nD. Mid palatine cyst\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0287", "text": "Compared to unfractionated heparin, Low molecular weight heparin (LMWH) has reliable anticoagulant action because:\nA. It interferes with thrombin and antithrombin III simultaneously\nB. It is less protein bound\nC. It is given subcutaneously\nD. It is cleared by macrophages\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0288", "text": "The neutralization of saliva is due to\nA. Mucin\nB. Ammonia\nC. Amino a cids\nD. Bicarbonates\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0289", "text": "In a patient of heart disease antibiotic prophylaxis for dental extraction is:\nA. Amoxicillin.\nB. Imipenem.\nC. Gentamicin.\nD. Erythromycin.\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0290", "text": "Ligamentum flavum consists of:\nA. Type-II collagen\nB. Type-III collagen\nC. Reticulin\nD. Elastin\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0291", "text": "Which of these is not a non-contraceptive use of levonorgestrel?\nA. Endometriosis\nB. Premenstrual tension\nC. Complex endometrial hJperplasia\nD. Emergencycontraception\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0292", "text": "A 38 years old female presented to the emergency with extensive burns. The patient had grade 3 burns on the face, back, upper arms and forearms along with singeing of hairs. Which of the following is not a proof of inhalation burns?\nA. Yellow colored sputum\nB. Blackish soot deposit on posterior pa of tongue\nC. Hoarseness & stridor of voice\nD. Singeing of eyebrows and facial hair\nAnswer with the letter of the correct option, then explain briefly.", "answer": "A"}, {"id": "medmcqa-dev-0293", "text": "In a\t6-month-old child, thick curd like white\tpatch\nappears on the buccal mucosa. On rubbing it leaves an\nerythematous patch. Most likely diagnosis is:\nA. Tuberculosis\nB. Lichen planus\nC. Lupus erythematous\nD. Candidiasis\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0294", "text": "Least pain sensitivity in oral cavity is in which mucosa\nA. Gingiva\nB. Buccal mucosa\nC. Labial mucosa\nD. Dorsal mucosa of tongue\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0295", "text": "An infant presented with the following lesions on his face and limbs. Which of the following is the most likely causative organism?\nA. HSV\nB. HPV\nC. Coxsackie\nD. CMV\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}, {"id": "medmcqa-dev-0296", "text": "Edge angle is the angle between:\nA. Rake face and radial line.\nB. Rake face and clearance face.\nC. Clearance face and tooth structure.\nD. Radial line and clearance face\nAnswer with the letter of the correct option, then explain briefly.", "answer": "B"}, {"id": "medmcqa-dev-0297", "text": "The facial proportions  between upper, middle and lower thirds of fare Is:\nA. 25:50:25\nB. 34:33:33\nC. 35:30:35\nD. 30:35:35\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0298", "text": "Mode:is defined as?\nA. Average of the values\nB. Relating to a value or quantity lying at the midpoint of a data after arranging in ascending or descending order\nC. Adding up the values and then dividing by the number of values\nD. The number which appears most often in a set of numbers\nAnswer with the letter of the correct option, then explain briefly.", "answer": "D"}, {"id": "medmcqa-dev-0299", "text": "Which of the following movement(s) is /are involved in the opening of the mouth?\nA. Only hinge (rotation)\nB. Only translation\nC. Hinge followed by translation\nD. Translation followed by hinge\nAnswer with the letter of the correct option, then explain briefly.", "answer": "C"}]''')
tok.padding_side = 'left'
letter_ids = {L: tok(f' {L}', add_special_tokens=False).input_ids[-1] for L in 'ABCD'}
preds, correct = [], 0
for i in range(0, len(DEV), 4):
    chunk = DEV[i:i+4]
    texts = [tok.apply_chat_template([{'role': 'user', 'content': r['text']}], tokenize=False, add_generation_prompt=True) + 'Answer:' for r in chunk]
    enc = tok(texts, return_tensors='pt', padding=True, add_special_tokens=False).to('cuda')
    with torch.inference_mode():
        logits = model(**enc).logits[:, -1, :].float()
    for r, lg in zip(chunk, logits):
        p = max('ABCD', key=lambda L: lg[letter_ids[L]].item()); preds.append(p); correct += (p == r['answer'])
RESULTS['medmcqa_dev300'] = {'accuracy': round(correct/len(DEV), 4), 'n': len(DEV), 'pred_dist': {L: preds.count(L) for L in 'ABCD'}}
RESULTS['medmcqa_preds'] = ''.join(preds)   # 300 letters, compared position by position with the laptop run
print('MedMCQA dev-300 accuracy on T4 FP16:', RESULTS['medmcqa_dev300'], '| laptop FP16: 0.4267')


MedMCQA dev-300 accuracy on T4 FP16: {'accuracy': 0.4233, 'n': 300, 'pred_dist': {'A': 122, 'B': 70, 'C': 88, 'D': 20}} | laptop FP16: 0.4267


In [5]:
# G.2 on the T4: NF4 weights, FP16 compute
del model; torch.cuda.empty_cache()
nf4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model4 = AutoModelForCausalLM.from_pretrained(name, quantization_config=nf4, device_map='cuda')
ids = tok(text, return_tensors='pt').to('cuda'); torch.cuda.reset_peak_memory_stats(); t0 = time.time()
out = model4.generate(**ids, max_new_tokens=128, do_sample=False); dt = time.time() - t0
RESULTS['g2_nf4'] = {'peak_gb': round(torch.cuda.max_memory_allocated()/1e9, 2), 'tok_per_s': round((out.shape[1]-ids.input_ids.shape[1])/dt, 1), 'answer_head': tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)[:200]}
print(json.dumps(RESULTS['g2_nf4'], ensure_ascii=False, indent=1))


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

{
 "peak_gb": 2.65,
 "tok_per_s": 12.3,
 "answer_head": "Paracetamol (Acetaminophen) aur BP (Blood Pressure) ki dawaon ka ek saath lena **safety** ka matter hai. Is ka sabab yeh hai ki:\n\n1. **Paracetamol** (4-6 mg/kg/day) aur **BP dhawo** (4-5 mg/kg/day) ka"
}


In [6]:
# Final block: copy everything between the markers back to the laptop session
RESULTS['versions'] = {'torch': torch.__version__, 'cuda': torch.version.cuda, 'transformers': transformers.__version__}
print('=====COLAB_RESULTS_BEGIN=====')
print(json.dumps(RESULTS, ensure_ascii=False))
print('=====COLAB_RESULTS_END=====')
json.dump(RESULTS, open('colab_results.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)  # also downloadable from the Files pane


=====COLAB_RESULTS_BEGIN=====
{"platform": "colab-t4", "preamble_present": true, "prompt_tokens": 379, "g1": {"peak_gb": 6.78, "tok_per_s": 16.0, "answer_head": "Paracetamol (Acetaminophen) और BP (Blood Pressure) की दवाओं के साथ लेने के बारे में सावधानी बरतने की आवश्यकता होती है। यहाँ कुछ महत्वपूर्ण बातें हैं जिन्हें ध्यान में रखना चाहिए:\n\n1. **संभावित इंटरैक्"}, "medmcqa_dev300": {"accuracy": 0.4233, "n": 300, "pred_dist": {"A": 122, "B": 70, "C": 88, "D": 20}}, "medmcqa_preds": "BABAACCCDBADBAAAAAACBDAAADBABAACBAAAAABCACCAADACAABBACAAAABAAACDCCAAACCBCABCACBAADCBCCBACCBCBAABCAACBCBBCACADBDCAABCAABABABABCCDABBABDCCBBCABDCCCDBAAACDDBABCAACCCCAACAABABBCCADBBCCABAAACACCAAACAABCABCABACAAACCAAACBBACAACBCBCCCBABCCACAAABDAADACCBCACACACABCBAABCBAABBCACACABABCBABBCCAACAABADACCABDCABBC", "g2_nf4": {"peak_gb": 2.65, "tok_per_s": 12.3, "answer_head": "Paracetamol (Acetaminophen) aur BP (Blood Pressure) ki dawaon ka ek saath lena **safety** ka matter hai. Is ka sabab yeh hai ki:\n\n1. **Paracetam